In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from keras.layers import Embedding, Input, dot, concatenate, Flatten
from keras.models import Model
from IPython.display import SVG
from keras.utils.vis_utils import model_to_dot

In [2]:
from tensorflow.python.client import device_lib

print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 425589635982197417
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 4162256896
locality {
  bus_id: 1
  links {
  }
}
incarnation: 10539537741984570742
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 2060, pci bus id: 0000:01:00.0, compute capability: 7.5"
xla_global_id: 416903419
]


In [3]:
import keras
import tensorflow as tf

config = tf.compat.v1.ConfigProto(device_count={'GPU': 1, 'CPU': 4})
sess = tf.compat.v1.Session(config=config)


In [4]:
keras.backend.set_session(sess)

#### Importing Movie-lens dataset

In [5]:
movies = pd.read_csv(
    'E:\\University Stuff\\Third year\\movie_recommender_research\\datasets\\movie_lens_2m\\movies.csv')
ratings = pd.read_csv(
    'E:\\University Stuff\\Third year\\movie_recommender_research\\datasets\\movie_lens_2m\\ratings.csv')

In [6]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,1112486027
1,1,29,3.5,1112484676
2,1,32,3.5,1112484819
3,1,47,3.5,1112484727
4,1,50,3.5,1112484580


In [7]:
ratings.drop(['timestamp'], axis=1, inplace=True)

In [8]:
df_combined = pd.merge(ratings, movies, on='movieId')

In [9]:
df_combined.head(100)

,userId,movieId,rating,title,genres
0,1,2,3.5,Jumanji (1995),Adventure|Children|Fantasy
1,5,2,3.0,Jumanji (1995),Adventure|Children|Fantasy
2,13,2,3.0,Jumanji (1995),Adventure|Children|Fantasy
3,29,2,3.0,Jumanji (1995),Adventure|Children|Fantasy
4,34,2,3.0,Jumanji (1995),Adventure|Children|Fantasy
...,...,...,...,...,...
95,611,2,5.0,Jumanji (1995),Adventure|Children|Fantasy
96,630,2,4.5,Jumanji (1995),Adventure|Children|Fantasy
97,635,2,4.0,Jumanji (1995),Adventure|Children|Fantasy
98,637,2,3.5,Jumanji (1995),Adventure|Children|Fantasy


#### Splitting Data into test and train

In [10]:
from sklearn.model_selection import train_test_split

X = ratings.iloc[:, :2]
Y = ratings.iloc[:, 2:]

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=66)

In [11]:
Y.head()
X.head()

,userId,movieId
0,1,2
1,1,29
2,1,32
3,1,47
4,1,50


### Embedding matrices

![alt text](https://i.imgur.com/zGQJFLD.png)

In [53]:
import tensorflow as tf
from tensorflow.python.keras import backend as K

# adjust values to your needs
config = tf.compat.v1.ConfigProto(device_count={'GPU': 1, 'CPU': 8})
sess = tf.compat.v1.Session(config=config)
K.set_session(sess)

In [16]:
# The number of latent factors for the embedding
n_latent_factors = 50

# no of users and movies
n_users = len(ratings['userId'].unique())
n_movies = len(ratings['movieId'].unique())

In [17]:
# Model Architecture


# User Embeddings
user_input = Input(shape=(1,), name='User_Input')
user_embeddings = Embedding(input_dim=n_users, output_dim=n_latent_factors, input_length=1,
                            name='User_Embedding')(user_input)
user_vector = Flatten(name='User_Vector')(user_embeddings)

# Movie Embeddings
movie_input = Input(shape=(1,), name='Movie_Input')
movie_embeddings = Embedding(input_dim=n_movies, output_dim=n_latent_factors, input_length=1,
                             name='Movie_Embedding')(movie_input)
movie_vector = Flatten(name='Movie_Vector')(movie_embeddings)

# Dot Product
merged_vectors = dot([user_vector, movie_vector], name='Dot_Product', axes=1)
model = Model([user_input, movie_input], merged_vectors)

### Compiling and Fitting the model

In [46]:
model.compile(loss='mean_squared_error', optimizer='Adam')

In [47]:
batch_size = 128
epochs = 20

history = model.fit(x=[x_train['userId'], x_train['movieId']],
                    y=y_train, batch_size=batch_size, epochs=epochs,
                    verbose=1, validation_data=([x_test['userId'], x_test['movieId']], y_test))

Epoch 1/20
125002/125002 [==============================] - 1224s 10ms/step - loss: 2.8996 - val_loss: 2.2620
Epoch 2/20
125002/125002 [==============================] - 1215s 10ms/step - loss: 2.1997 - val_loss: 2.1917
Epoch 3/20
125002/125002 [==============================] - 1235s 10ms/step - loss: 2.1155 - val_loss: 2.1816
Epoch 4/20
125002/125002 [==============================] - 1176s 9ms/step - loss: 2.0673 - val_loss: 2.1870
Epoch 5/20
125002/125002 [==============================] - 1293s 10ms/step - loss: 2.0382 - val_loss: 2.1951
Epoch 6/20
125002/125002 [==============================] - 1247s 10ms/step - loss: 2.0198 - val_loss: 2.2033
Epoch 7/20
125002/125002 [==============================] - 1151s 9ms/step - loss: 2.0072 - val_loss: 2.2137
Epoch 8/20
125002/125002 [==============================] - 1206s 10ms/step - loss: 1.9982 - val_loss: 2.2211
Epoch 9/20
125002/125002 [==============================] - 1285s 10ms/step - loss: 1.9916 - val_loss: 2.2289
Epoch 10/20


In [49]:
model.save('E:\\University Stuff\\Third year\\movie_recommender_research\\Models\\DNN recommender')

INFO:tensorflow:Assets written to: E:\University Stuff\Third year\movie_recommender_research\Models\DNN recommender\assets


INFO:tensorflow:Assets written to: E:\University Stuff\Third year\movie_recommender_research\Models\DNN recommender\assets


In [19]:
score = model.evaluate([x_test['userId'], x_test['movieId']], y_test)
print()
print('RMSE: {:.4f}'.format(np.sqrt(score)))

125002/125002 [==============================] - 309s 2ms/step - loss: 2.2935

RMSE: 1.5144


### Import Testing dataset

ML-1M will be imported for testing purposes to mitigate RAM insufficiency

In [3]:
# import pandas as pd
# ratings = pd.read_csv('E:\\University Stuff\\Third year\\movie_recommender_research\\datasets\\movie_lens_1m\\ratings.dat', sep='::', engine='python')
# ratings.drop(['978300760'], axis=1, inplace=True)
# ratings = ratings.rename(columns = {'1' : 'userId', '1193' : 'movieId', '5' : 'rating' })
# print('done')

done
done
done


In [4]:
print(ratings.columns)

Index(['userId', 'movieId', 'rating'], dtype='object')


In [10]:
from surprise import NormalPredictor
from surprise import Dataset
from surprise import Reader
from surprise.model_selection import cross_validate
import keras

# A reader is still needed but only the rating_scale param is required.
reader = Reader(rating_scale=(1, 5))
# The columns must correspond to user id, item id and ratings (in that order).
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

# Load Model
model = keras.models.load_model('E:\\University Stuff\\Third year\\movie_recommender_research\\Models\\DNN recommender')

# We can now use this dataset as we please, e.g. calling cross_validate
cross_validate(model, data, measures=['RMSE', 'MAE'], cv=3, verbose=True, n_jobs=-1)

INFO:tensorflow:Assets written to: ram://bb1066b5-134d-4a9b-a7fe-fff93222f1d6/assets


INFO:tensorflow:Assets written to: ram://bb1066b5-134d-4a9b-a7fe-fff93222f1d6/assets


INFO:tensorflow:Assets written to: ram://0f8109a5-bdab-46da-8ace-82e8ddf5ebd2/assets


INFO:tensorflow:Assets written to: ram://0f8109a5-bdab-46da-8ace-82e8ddf5ebd2/assets


BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.